# Layer 0 Linear Probing Skyline

This notebook is intentionally simple:
1) set target + layer,
2) load `X`/`y`,
3) run all linear probes,
4) show/save skyline.

It uses the refactored probing code only.


In [1]:
from pathlib import Path
import os
import pandas as pd

from prob.paths_and_io import get_project_root, get_exp_dirs, load_X_from_pt, load_y_by_ids
from prob.prob_config import get_ds_load_config
from prob.prob_registry import LINEAR_PROBES
from prob.prob_run import run_cv_search


In [2]:
# Ensure stable root discovery for this notebook session.
project_root = get_project_root()
os.environ["HOME_PROJ_DIR"] = str(project_root)
print("Project root:", project_root)

# --- only edit these ---
TARGET_FILE = "weighted_hb_score.pt"
LAYER_NUM = 0

# Runtime knobs
RANDOM_STATE = 96
N_SPLITS_CV = 5
TEST_SIZE = 0.1

# Cluster-safe CPU budget:
# - Uses scheduler-provided CPU count when available
# - Falls back to 1 core (safe default)
CPU_BUDGET = int(
    os.getenv("SLURM_CPUS_PER_TASK")
    or os.getenv("NSLOTS")
    or os.getenv("OMP_NUM_THREADS")
    or 1
)

# Keep BLAS/OpenMP single-threaded per worker to avoid oversubscription.
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

N_JOBS = CPU_BUDGET
print(f"CPU_BUDGET={CPU_BUDGET}, N_JOBS={N_JOBS}")

# Reuse tuned params for similar reruns.
REUSE_BEST_PARAMS = True
BEST_PARAMS_CACHE_DIR = project_root / "data" / "probing" / "_shared_best_params"


Project root: /home/famo00001/kinodata-3D-affinity-prediction
CPU_BUDGET=16, N_JOBS=16


In [3]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

# Important:
# - Linear probing here uses scikit-learn models (Ridge/Lasso), which run on CPU.
# - GPU helps when generating GNN representations, not in this sklearn probing step.
# - To speed this notebook, increase CPU parallelism via N_JOBS.


CUDA available: True
GPU count: 1


In [4]:
prob_config = get_ds_load_config(
    target_file=TARGET_FILE,
)

X = load_X_from_pt(prob_config.output_dir, layer_num=LAYER_NUM)
y = load_y_by_ids(
    prob_config.output_dir,
    target_dir=prob_config.target_dir,
    targets_file=TARGET_FILE,
)

target_name = Path(TARGET_FILE).stem
print("X shape:", X.shape)
print("y shape:", y.shape)
print("target:", target_name)


X shape: (41238, 256)
y shape: (41238,)
target: weighted_hb_score


In [5]:
# let's just do ridge for now
# LINEAR_PROBES.pop(-1)
LINEAR_PROBES

[{'name': 'ridge',
  'estimator': Ridge(random_state=96),
  'param_grid': {'model__alpha': array([1.e-05, 1.e-04, 1.e-03, 1.e-02, 1.e-01, 1.e+00, 1.e+01, 1.e+02,
          1.e+03, 1.e+04])}},
 {'name': 'lasso',
  'estimator': Lasso(max_iter=10000, random_state=96),
  'param_grid': {'model__alpha': array([1.e-05, 1.e-04, 1.e-03, 1.e-02, 1.e-01, 1.e+00, 1.e+01])}}]

In [6]:
runs = []

for entry in LINEAR_PROBES:
    name = entry["name"]
    estimator = entry["estimator"]
    param_grid = entry.get("param_grid", {})

    exp_dirs = get_exp_dirs(
        prob_config.output_dir,
        target=target_name,
        prob_model=name,
        layer_num=LAYER_NUM,
    )

    search, metrics, _ = run_cv_search(
        X,
        y,
        estimator,
        param_grid,
        n_splits=N_SPLITS_CV,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        n_jobs=N_JOBS,
        exp_dirs=exp_dirs,
        model_name=name,
        run_stats=True,
        reuse_best_params=REUSE_BEST_PARAMS,
        best_params_cache_dir=BEST_PARAMS_CACHE_DIR,
    )

    runs.append({
        "model": name,
        "layer": LAYER_NUM,
        **metrics,
        **search.best_params_,
    })

results_df = pd.DataFrame(runs)
results_df


Fitting 5 folds for each of 7 candidates, totalling 35 fits


/opt/conda/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:697: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 6.014e+04, tolerance: 5.261e+01
  model = cd_fast.enet_coordinate_descent(
/opt/conda/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:697: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 6.664e+04, tolerance: 5.290e+01
  model = cd_fast.enet_coordinate_descent(
/opt/conda/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:697: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.772e+04, tolerance: 5.293e

,model,layer,r2,rmse,mae,fit_seconds,model__alpha
0,ridge,0,0.128585,3.942912,3.142151,0.475741,0.00100
1,lasso,0,0.079497,4.052444,3.241822,93.974586,0.00001


In [7]:
# Skyline: higher R2 is better, lower RMSE is better.
skyline_df = (
    results_df
    .sort_values(["r2", "rmse"], ascending=[False, True])
    .reset_index(drop=True)
)

display(skyline_df[["model", "r2", "rmse", "mae", "fit_seconds"]])
best_model = skyline_df.iloc[0]
print(f"Best skyline model: {best_model['model']} | R2={best_model['r2']:.4f} | RMSE={best_model['rmse']:.4f}")


,model,r2,rmse,mae,fit_seconds
0,ridge,0.128585,3.942912,3.142151,0.475741
1,lasso,0.079497,4.052444,3.241822,93.974586


Best skyline model: ridge | R2=0.1286 | RMSE=3.9429


In [8]:
# Save notebook-level skyline summary.
skyline_out = Path(prob_config.output_dir) / target_name / "experiments" / f"layer_{LAYER_NUM}_linear_skyline.csv"
skyline_out.parent.mkdir(parents=True, exist_ok=True)
skyline_df.to_csv(skyline_out, index=False)
print("Saved skyline:", skyline_out)


Saved skyline: /home/famo00001/kinodata-3D-affinity-prediction/data/probing/CGNN-3D/rmsd_cutoff_2/random-k-fold/weighted_hb_score/experiments/layer_0_linear_skyline.csv
